# AutoPose — GigaPose-style 6-DoF Object Pose Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunachaleshwaran/autopose/blob/main/notebooks/autopose_colab.ipynb)

## Pipeline Overview

| Stage | What | How |
|---|---|---|
| **1 — Template retrieval** | Predict azimuth + elevation (2 DoF) | Nearest-neighbour search in Fae (DINOv2) feature space over 162 icosphere templates |
| **2 — In-plane refinement** | Predict in-plane rotation + translation + scale (4 DoF) | Fist (ResNet18 + MLP heads) regresses (cos α, sin α, tx, ty, s) |

Azimuth & elevation are **retrieved, not regressed** — the best-matching template's pose from `templates/poses.json` is the answer.

---

**Before running:** upload your local `templates/` folder (162 PNGs + `poses.json`) to Google Drive at `MyDrive/autopose/templates/`.

## 0 — Setup

In [ ]:
# Clone repo (skip if already present)
import os

REPO_DIR = "/content/autopose"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/arunachaleshwaran/autopose.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

In [ ]:
# System dependencies (pyrender needs osmesa for headless rendering)
import os
import subprocess

subprocess.run(["apt-get", "install", "-qq", "libosmesa6"], check=True)
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("System deps installed.")

In [ ]:
# Install pip packages from environment.yml (pip: section)
import subprocess
import yaml

with open("environment.yml") as f:
    env = yaml.safe_load(f)

pip_pkgs = next(
    d["pip"] for d in env["dependencies"] if isinstance(d, dict) and "pip" in d
)

print(f"Installing {len(pip_pkgs)} packages from environment.yml pip section...")
subprocess.check_call(["pip", "install", "-q"] + pip_pkgs)
print("Done.")

In [ ]:
# Install conda-listed packages not already in Colab base
# (torch/numpy/scipy/pandas/matplotlib are pre-installed on Colab)
!pip install -q \
    opencv-python-headless \
    trimesh \
    networkx \
    shapely \
    imageio \
    scikit-image \
    tqdm \
    ipywidgets
print("Conda-equivalent packages installed.")

In [ ]:
# Mount Google Drive and symlink templates/
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/autopose"
TEMPLATES_SRC = f"{DRIVE_ROOT}/templates"
TEMPLATES_DST = "/content/autopose/templates"

assert os.path.isdir(TEMPLATES_SRC), (
    f"templates/ not found at {TEMPLATES_SRC}. "
    "Upload your templates/ folder to Google Drive at MyDrive/autopose/templates/"
)

# Symlink so all downstream code uses templates/ as a normal relative path
if not os.path.islink(TEMPLATES_DST):
    os.symlink(TEMPLATES_SRC, TEMPLATES_DST)

import json
poses = json.loads(open("templates/poses.json").read())
print(f"templates/ linked. Found {len(poses)} pose entries.")

In [ ]:
# Verify GPU and key imports
import torch
import timm
import lightning
import transformers
import kornia
import roma
import trimesh
import pyrender

print(f"torch      {torch.__version__}")
print(f"lightning  {lightning.__version__}")
print(f"timm       {timm.__version__}")
print(f"transformers {transformers.__version__}")
print()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — go to Runtime > Change runtime type > GPU")

---

## 1 — Load Template Data

`templates/poses.json` stores 162 entries, one per icosphere vertex. Each entry has:
- `azimuth_deg` / `elevation_deg` — the viewpoint angle
- `R_world_to_cam_3x3` — rotation matrix (world → camera)
- `t_cam_to_world_3` — camera position in world space
- `file` — corresponding PNG (`tmpl_XXXX.png`)

Azimuth + elevation together define the **out-of-plane rotation** (2 of 6 DoF).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

poses = json.loads(open("templates/poses.json").read())
df = pd.DataFrame(poses)
print(f"Loaded {len(df)} templates")
display(df[["frame", "file", "azimuth_deg", "elevation_deg"]].head())

def load_template_rgb(path):
    """Load RGBA template PNG and composite onto white background."""
    img = Image.open(path).convert("RGBA")
    bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
    bg.paste(img, mask=img.split()[3])
    return bg.convert("RGB")

templates_rgb = [load_template_rgb(f"templates/{row['file']}") for _, row in df.iterrows()]
print(f"Loaded {len(templates_rgb)} template images  size={templates_rgb[0].size}")

# Viewpoint sphere
fig = plt.figure(figsize=(10, 4))
ax3d = fig.add_subplot(121, projection="3d")
xyz = np.array(df["viewpoint_xyz"].tolist())
ax3d.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], s=10, c=df["elevation_deg"], cmap="coolwarm")
ax3d.set_title("162 icosphere viewpoints")

# Az/El scatter
ax2d = fig.add_subplot(122)
sc = ax2d.scatter(df["azimuth_deg"], df["elevation_deg"], s=15, c=df["elevation_deg"], cmap="coolwarm")
ax2d.set_xlabel("Azimuth (°)"); ax2d.set_ylabel("Elevation (°)")
ax2d.set_title("Azimuth vs Elevation"); plt.colorbar(sc, ax=ax2d, label="elevation °")
plt.tight_layout(); plt.show()

# Grid of 12 sample templates
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
indices = np.linspace(0, 161, 12, dtype=int)
for ax, i in zip(axes.flat, indices):
    ax.imshow(templates_rgb[i])
    ax.set_title(f"az={df.iloc[i]['azimuth_deg']:.0f}°\nel={df.iloc[i]['elevation_deg']:.0f}°", fontsize=7)
    ax.axis("off")
plt.suptitle("Sample templates"); plt.tight_layout(); plt.show()

---

## 2 — Template Feature Bank (Fae)

We use a **pretrained DINOv2 ViT-L/14** (no fine-tuning) as Fae. The paper fine-tunes it with InfoNCE loss, but pretrained DINOv2 still retrieves meaningful viewpoints and is immediately runnable.

- Input: 224×224 RGB image
- Patch size: 14×14 px → **16×16 = 256 patches** per image
- Output per patch: **1024-dim** feature vector
- Total template bank shape: **(162, 256, 1024)**

All 162 template features are precomputed once and saved to `templates/template_features.pt`.

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# DINOv2 ViT-L/14 — 1024-dim patch tokens, matches paper's 16×16×1024 output
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
dino = dino.eval().to(DEVICE)
print("DINOv2 ViT-L/14 loaded")

# ImageNet normalisation (same stats DINOv2 was pretrained with)
preprocess = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Batch forward pass over all 162 templates
BATCH = 16
all_feats = []

with torch.no_grad():
    for i in range(0, len(templates_rgb), BATCH):
        batch = torch.stack([preprocess(img) for img in templates_rgb[i:i+BATCH]]).to(DEVICE)
        # x_norm_patchtokens: (B, num_patches, dim) = (B, 256, 1024)
        out = dino.forward_features(batch)["x_norm_patchtokens"]
        all_feats.append(out.cpu())
        print(f"  batch {i//BATCH + 1}/{-(-len(templates_rgb)//BATCH)}  shape={tuple(out.shape)}", end="\r")

template_feats = torch.cat(all_feats, dim=0)          # (162, 256, 1024)
template_feats = F.normalize(template_feats, dim=-1)   # L2-normalise for cosine sim

torch.save(template_feats, "templates/template_features.pt")
print(f"\nSaved templates/template_features.pt  shape={tuple(template_feats.shape)}")

---

## 3 — Azimuth & Elevation Prediction (Stage 1)

**How it works:**
1. Upload a real photo of your 3DBenchy and set a manual crop box
2. The crop is encoded by DINOv2 → `(256, 1024)` patch features
3. Each query patch finds its nearest match across all `162 × 256` template patches (cosine similarity)
4. Templates are scored by the mean similarity of patches assigned to them (threshold 0.5)
5. The top-K templates' `azimuth_deg` / `elevation_deg` from `poses.json` **are** the prediction

In [ ]:
### 3a — Upload photo and set crop box

from google.colab import files
import io

uploaded = files.upload()
fname = next(iter(uploaded))
query_img = Image.open(io.BytesIO(uploaded[fname])).convert("RGB")
W, H = query_img.size
print(f"Image size: {W}×{H}")

plt.figure(figsize=(6, 6))
plt.imshow(query_img); plt.axis("off")
plt.title("Full query image — note pixel coordinates for crop below")
plt.show()

# ── Edit these four values to frame the 3DBenchy ──────────────────────────────
X1, Y1 = 100,  80   # top-left  corner (pixels)
X2, Y2 = 520, 460   # bottom-right corner (pixels)
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib.patches as patches
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(query_img)
rect = patches.Rectangle((X1, Y1), X2-X1, Y2-Y1, linewidth=2, edgecolor="lime", facecolor="none")
axes[0].add_patch(rect); axes[0].set_title("Crop box"); axes[0].axis("off")

crop = query_img.crop((X1, Y1, X2, Y2))
axes[1].imshow(crop); axes[1].set_title(f"Query crop  {crop.size[0]}×{crop.size[1]}px"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
### 3b — Extract query features

query_tensor = preprocess(crop).unsqueeze(0).to(DEVICE)   # (1, 3, 224, 224)

with torch.no_grad():
    query_feats = dino.forward_features(query_tensor)["x_norm_patchtokens"]  # (1, 256, 1024)

query_feats = F.normalize(query_feats.squeeze(0), dim=-1)  # (256, 1024)
print(f"Query features: {tuple(query_feats.shape)}")

In [ ]:
### 3c — Per-patch nearest-neighbour retrieval + template scoring

# Load cached template features (skip if already in memory from Section 2)
if "template_feats" not in dir():
    template_feats = torch.load("templates/template_features.pt")

tf = template_feats.to(DEVICE)    # (162, 256, 1024)
qf = query_feats.to(DEVICE)       # (256, 1024)

# Flatten all template patches: (162*256, 1024)
tf_flat = tf.view(162 * 256, 1024)

# Cosine similarity: each query patch vs every template patch
# Both are L2-normalised → dot product = cosine sim
sim_matrix = qf @ tf_flat.T       # (256, 41472)

# For each query patch: best matching template patch (globally)
best_sim, best_idx = sim_matrix.max(dim=1)     # (256,), (256,)
best_template = best_idx // 256                # which of 162 templates

# Apply 0.5 threshold (paper Eq. 2)
keep = best_sim >= 0.5
kept_sims = best_sim[keep]
kept_templates = best_template[keep]
print(f"{keep.sum().item()} / 256 query patches passed the 0.5 threshold")

# Score each template: mean similarity over its assigned query patches (paper Eq. 3)
scores = torch.zeros(162, device=DEVICE)
counts = torch.zeros(162, device=DEVICE)
scores.scatter_add_(0, kept_templates, kept_sims)
counts.scatter_add_(0, kept_templates, torch.ones_like(kept_sims))
template_scores = scores / counts.clamp(min=1)

# Top-5 results
K = 5
topk = template_scores.topk(K)
print(f"\nTop-{K} retrieved templates:")
for rank, (score, idx) in enumerate(zip(topk.values, topk.indices), 1):
    row = df.iloc[idx.item()]
    print(f"  #{rank}  template {idx.item():3d}  score={score:.3f}  "
          f"az={row['azimuth_deg']:+7.1f}°  el={row['elevation_deg']:+6.1f}°")

In [ ]:
### 3d — Visualisation

best_i = topk.indices[0].item()
best_row = df.iloc[best_i]

fig, axes = plt.subplots(1, K + 1, figsize=(3 * (K + 1), 3))

axes[0].imshow(crop)
axes[0].set_title("Query crop", fontsize=9)
axes[0].axis("off")

for rank, idx in enumerate(topk.indices, 1):
    row = df.iloc[idx.item()]
    axes[rank].imshow(templates_rgb[idx.item()])
    axes[rank].set_title(
        f"#{rank}  score={template_scores[idx]:.2f}\n"
        f"az={row['azimuth_deg']:+.0f}°  el={row['elevation_deg']:+.0f}°",
        fontsize=8,
    )
    axes[rank].axis("off")

plt.suptitle("Query  →  Top-5 retrieved templates", fontsize=11)
plt.tight_layout(); plt.show()

print(f"\nPredicted viewpoint:")
print(f"  Azimuth  : {best_row['azimuth_deg']:+.1f}°")
print(f"  Elevation: {best_row['elevation_deg']:+.1f}°")
print(f"  R_world_to_cam:\n{np.array(best_row['R_world_to_cam_3x3'])}")

---

## 4 — In-Plane Refinement (Fist, Stage 2)

Once the out-of-plane rotation (az + el) is fixed from Stage 1, **Fist** (ResNet18 + MLP heads) regresses the remaining 4 DoF:
- `(cos α, sin α)` — in-plane rotation
- `(tx, ty)` — 2D translation
- `s` — scale

Input: query crop + matched template crop (concatenated along channel dim).

In [ ]:
# TODO: define Fist — ResNet18 backbone + 3 MLP heads (rotation, translation, scale)
# TODO: input: concat(query_crop, template_crop) → (B, 6, H, W)
# TODO: output: (cos_alpha, sin_alpha, tx, ty, s)

---

## 5 — Training

Two separate training runs:
1. **Fae** — InfoNCE contrastive loss. Positives: same az/el, different in-plane.
2. **Fist** — L1/L2 regression loss on `(cos α, sin α, tx, ty, s)` given Stage 1 predictions.

Use PyTorch Lightning `Trainer` with `wandb` logging.

In [ ]:
# TODO: LightningDataModule — load rendered templates + online augmentation (albumentations)
# TODO: FaeLightningModule — InfoNCE loss, configure_optimizers
# TODO: FistLightningModule — regression loss, configure_optimizers
# TODO: trainer = Trainer(max_epochs=100, accelerator='gpu', logger=WandbLogger(...))
# TODO: trainer.fit(model, datamodule)